# Schedule Adherence at Timepoint

### Definition
At each transit stop, the proportion of trips that are less than five minutes late and less than one minute early.

### Reporting Grain
Reporting Period and Day Part. (also mapped at timepoints here)

# Average Deviation from Schedule

### Definition

The difference between when the bus is scheduled to arrive at each stop and when the bus arrives based on real time data. Reported as 50th and 80th percentile.



In [ ]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely.wkt

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_feed_entries,
    get_schedule_rt_stop_times,
)
from src.metric_calculation import calculate_on_time_performance
from src.secrets import carto_map_parameters

### Parameters

In [ ]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

In [ ]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

# beat the published arrival by this or more and the vehicle is early; miss it
# by this or more and it is late. Both inclusive
EARLY_SECONDS = 60
LATE_SECONDS = 5 * 60

### Implementation

In [ ]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))

In [ ]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

In [ ]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

In [ ]:
# One row per trip and timepoint: when the vehicle actually arrived, and the
# arrival dim_stop_arrivals published for that trip at that stop. This is the
# same frame trip_duration_variability.ipynb reads - that metric uses only the
# realtime columns, this one compares the two arrivals
schedule_rt_stop_times = get_schedule_rt_stop_times(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
schedule_rt_stop_times

In [ ]:
# How much of the realtime data found a published arrival to be measured
# against. The join is a left one, so an unmatched trip arrives here as a null
# deviation rather than dropping out - the on time metric drops those, the
# travel time metric keeps them
n_arrivals = len(schedule_rt_stop_times)
n_matched = schedule_rt_stop_times["arrival_deviation_seconds"].notna().sum()
print(f"{n_matched:,} of {n_arrivals:,} timepoint arrivals matched a published arrival "
      f"({n_matched / n_arrivals:.1%})")

# the deviation before any bucketing, in minutes. A long tail either side is
# worth a look before trusting the proportions: a cluster near a whole number of
# hours usually means a timezone or a GTFS-hours-past-24 problem rather than
# service running that badly
(schedule_rt_stop_times["arrival_deviation_seconds"]).describe()

### Results

In [ ]:
# Both tables at once, pooled over every selected date.
#
# by_stop says where along the line punctuality goes: a trip that leaves on time
# and loses six minutes by the end of the run shows up at the last timepoints
# only. by_period says how the whole pattern did in a day part - its percentiles
# are taken over the arrivals rather than over the per-stop numbers, so they are
# not the average of the table above.
on_time = calculate_on_time_performance(
    schedule_rt_stop_times,
    early_seconds=EARLY_SECONDS,
    late_seconds=LATE_SECONDS,
)
on_time.by_period

In [ ]:
on_time.by_stop

In [ ]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by the share of arrivals that were not on time.
MAP_TIME_PERIOD = "Midday"

# The stop times query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_stop_times[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    on_time.by_stop.loc[on_time.by_stop["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# the proportions are 0 to 1; the map reads better in percent. The colour and the
# tooltip use the one column, so the legend matches what is hovered
for column in [
    "proportion_early",
    "proportion_late",
    "proportion_not_on_time",
    "schedule_adherence",
]:
    stops_for_period[column] = (stops_for_period[column] * 100).round(1)
for column in [
    "p50_absolute_deviation_seconds",
    "p80_absolute_deviation_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
period_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    **carto_map_parameters(),
    name="service patterns",
)
stops_for_period.explore(
    m=period_map,
    column="proportion_not_on_time",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_timepoint_arrivals",
        "p50_absolute_deviation_seconds",
        "p80_absolute_deviation_seconds",
        "proportion_early",
        "proportion_late",
        "proportion_not_on_time",
        "schedule_adherence",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Arrivals",
            "Median deviation (s)",
            "80th pct deviation (s)",
            "% early",
            "% late",
            "% not on time",
            "% adherence",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} arrivals not on time"
    },
    name=f"{MAP_TIME_PERIOD} on time performance",
)
period_map.save(f"on_time_performance_{MAP_TIME_PERIOD.lower()}.html")
period_map